# Solution C2 · Encoding a nominal with integers: the invented order

**Type**: Break the model

## Objective

Show that a linear model with a nominal variable encoded as integers depends on the **arbitrary order** of the
categories, while the one-hot does not, and that a tree copes because it isolates the categories with its splits.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
n = 1200
effect = {"A": 6, "B": -6, "C": 5, "D": -5, "E": 7, "F": -7, "G": 4, "H": -4}      # high and low alternate: non-monotonic
df = pd.DataFrame({"zone": rng.choice(list(effect), n), "x": rng.normal(size=n)})
y = df.zone.map(effect).to_numpy() + 2 * df.x.to_numpy() + rng.normal(size=n)
categories = np.array(sorted(effect))


def evaluate(encoder, model):
    pipe = make_pipeline(ColumnTransformer([("zone", encoder, ["zone"])], remainder="passthrough"), model)
    return cross_val_score(pipe, df[["zone", "x"]], y, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()

## Commented solution

In [ ]:
r2_ord_lin = evaluate(OrdinalEncoder(), LinearRegression())
r2_ohe_lin = evaluate(OneHotEncoder(sparse=False), LinearRegression())
tree = lambda: DecisionTreeRegressor(max_depth=8, random_state=0)
r2_ord_tree = evaluate(OrdinalEncoder(), tree())
r2_ohe_tree = evaluate(OneHotEncoder(sparse=False), tree())
print(f"linear:  ordinal {r2_ord_lin:.3f} | one-hot {r2_ohe_lin:.3f}")
print(f"tree  :  ordinal {r2_ord_tree:.3f} | one-hot {r2_ohe_tree:.3f}")

In [ ]:
r2_perm = []
for s in range(200):
    order = np.random.default_rng(s).permutation(categories)
    r2_perm.append(evaluate(OrdinalEncoder(categories=[order]), LinearRegression()))
r2_perm = np.array(r2_perm)
spread = r2_perm.max() - r2_perm.min()
r2_perm_max = r2_perm.max()
print(f"R2 with random order: minimum {r2_perm.min():.3f} | median {np.median(r2_perm):.3f} | maximum {r2_perm_max:.3f}")

In [ ]:
answer = "b"

## Check

In [ ]:
with verify():
    check(r2_ohe_lin > 0.85, "the one-hot with the linear model recovers the effect of the zones (R2 > 0.85)", "the linear one-hot should have a high R2")
    check(r2_ord_lin < 0.3, "with integer codes the linear model fails (R2 < 0.3)", "with alphabetical order the effect is not monotonic")
    check(r2_ohe_lin > r2_ord_lin + 0.5, "huge difference between the two encodings, on the same data", "r2_ohe_lin must exceed r2_ord_lin by more than 0.5")
    check(r2_ord_tree > 0.8, "the tree copes even with integer codes (it isolates the categories with its splits)", "a depth 8 tree should have R2 > 0.8")
    check(len(r2_perm) == 200 and spread > 0.15, "changing ONLY the order of the categories the R2 varies a lot", "the spread between the permutations should exceed 0.15")
    check(r2_ohe_lin > r2_perm_max + 0.05 and np.median(r2_perm) < 0.3,
          "the one-hot beats even the best random order, and a typical order is terrible (median below 0.3)",
          "the one-hot must exceed the maximum of the permutations, whose median is low")
    check(answer == "b", "correct explanation", "reread: what does a linear model assume about the numeric code?")
done()

## What you learned

- With integer codes the linear model estimates **a single slope** per zone: the effect is forced to grow (or fall)
  linearly along the chosen order. With a non-monotonic effect the R² is close to zero, with the one-hot it is about 0.9.
- The result depends on the **order** of the categories, that is on an arbitrary choice (here the alphabetical order). Among the 200
  permutations the R² goes from about 0.1 to about 0.88 (median 0.17): the best order puts the "high" zones together and the "low" ones, but you
  know it only by looking at the target, and anyway it does not reach the one-hot.
- A tree uses the splits to isolate the categories: with enough depth it does not see the difference between the two
  encodings. It is the reason why trees and boosting are tolerant of ordinal encoding (it is not a rule: with
  many categories you need many splits).

## Common pitfall

Using `LabelEncoder` or `OrdinalEncoder` "because it is more convenient" on a nominal and then passing the data to a linear model, a
KNN or a network. No error, no warning: just a worse model, and nobody knows why.

## Interview question

<details><summary>When is the OrdinalEncoder acceptable on a nominal variable?</summary>

With tree models (trees, forests, boosting): the model works with thresholds and can isolate a category with two splits,
so it is not bound to the order. With few categories and enough data it works; with many categories you need many splits and
a better encoding is preferable (target encoding, native handling of categoricals). With linear models, KNN, SVM or networks
it is wrong: you need the one-hot or another encoding.

</details>